# INFO-F-422 — Task 2b: MIMO & Multi-Step-Ahead Prediction Strategy

## Context and Problem Statement

Prediction on the test dataset raises two major challenges:

1. **MIMO** (Multi-Input Multi-Output): the outputs $y_1$ and $y_2$ are coupled — each depends on the history of the other.
2. **Multi-step-ahead**: we must predict **1000 steps** ahead, without ever having access to the true values of $y$ during inference. The model must therefore reuse its own predictions as inputs, which creates a risk of **error explosion**.



---
## 1. Handling MIMO

The two-output NARX system is described by:

$$y_1(k+1) = f_1\bigl(y_1(k-d),\ldots,y_2(k-d-n_a),\; u_1(k),\ldots,u_2(k-n_b)\bigr) + w_1(k+1)$$
$$y_2(k+1) = f_2\bigl(y_1(k-d),\ldots,y_2(k-d-n_a),\; u_1(k),\ldots,u_2(k-n_b)\bigr) + w_2(k+1)$$

### Two possibles approaches

#### Approach A — Two independents models (Single-Output)
We train a model $M_1$ for $y_1$ and a model $M_2$ for $y_2$, both using the same feature vector $\phi(k)$.

**Advantages**: simple, each model is optimised for its own target.  
**Disadvantages**: the two models are trained independently, without sharing information about the correlation between $y_1$ and $y_2$.

#### Approach B — A single Multi-Output model
We train a single model $M$ that produces $(\hat{y}_1, \hat{y}_2)$ simultaneously.

**Advantages**: exploits the correlation between outputs, a single pipeline.  
**Disadvantages**: more complex to tune; some sklearn models do not natively support multi-output.

### Chosen strategy: Approach A (two independent models)

The NARX equations show that $f_1$ and $f_2$ are distinct functions. Training two separate models is a natural decomposition of the problem. The coupling between $y_1$ and $y_2$ is captured **implicitly** via the shared feature vector $\phi(k)$ (which contains lags of both outputs).  
This approach is also more flexible: different hyperparameters can be chosen for $M_1$ and $M_2$.